<a href="https://colab.research.google.com/github/ksehar99/EyePACS-DL-Blockchain/blob/main/DR_DL_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from PIL import Image
import numpy as np

# ── Setup ──────────────────────────────────────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/EyePACS/EyePACS"
TARGET_SIZE = (224, 224)   # MobileNetV2 ka required input size
BATCH_SIZE = 32

# ── Step A: Folder check ───────────────────
def check_data(base_dir):
    info = {}
    # Update the splits list to correctly handle 'test_merge' as a single directory
    splits_to_check = ["train", "valid"]
    test_split_path = os.path.join("test", "test_merge") # Path to the actual image directory

    for split_dir in splits_to_check: # For train and valid, we expect subfolders
        for cls in ["Nrdr", "Rdr"]:
            path = os.path.join(base_dir, split_dir, cls)
            if os.path.exists(path):
                files = os.listdir(path)
                info[f"{split_dir}/{cls}"] = len(files)
                if files:
                    sample = Image.open(os.path.join(path, files[0]))
                    print(f"{split_dir}/{cls}: {len(files)} images | sample size: {sample.size} | mode: {sample.mode}")
            else:
                print(f"{split_dir}/{cls}: PATH NOT FOUND")

    # Handle the 'test_merge' directory separately
    path = os.path.join(base_dir, test_split_path)
    if os.path.exists(path):
        files = os.listdir(path)
        # Filter for image files only
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
        info[test_split_path] = len(image_files)
        if image_files:
            sample = Image.open(os.path.join(path, image_files[0]))
            print(f"{test_split_path}: {len(image_files)} images | sample size: {sample.size} | mode: {sample.mode} (Labels inferred from filenames)")
        else:
            print(f"{test_split_path}: 0 image files found.")
    else:
        print(f"{test_split_path}: PATH NOT FOUND")
    return info

print("Checking dataset...\n")
data_info = check_data(BASE_DIR)


Checking dataset...

train/Nrdr: 2100 images | sample size: (1024, 1024) | mode: RGB
train/Rdr: 2100 images | sample size: (1024, 936) | mode: RGB
valid/Nrdr: 600 images | sample size: (1024, 808) | mode: RGB
valid/Rdr: 600 images | sample size: (1024, 1024) | mode: RGB
test/test_merge: 600 images | sample size: (1024, 823) | mode: RGB (Labels inferred from filenames)


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

BASE_DIR    = "/content/drive/MyDrive/EyePACS/EyePACS"
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS_P1   = 20
EPOCHS_P2   = 15
SEED        = 42

In [ ]:
import os
import tensorflow as tf
import numpy as np

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "train"),
    labels="inferred",
    label_mode="binary",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "valid"),
    labels="inferred",
    label_mode="binary",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Custom loading for test_ds_raw as labels are in filenames within test/test_merge
test_merge_path = os.path.join(BASE_DIR, "test", "test_merge")

image_paths = []
labels = []

# Collect image paths and infer labels from filenames
# Nrdr (0) for levels 0 and 1, Rdr (1) for levels 2, 3, 4
if os.path.exists(test_merge_path):
    for filename in os.listdir(test_merge_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            full_path = os.path.join(test_merge_path, filename)
            image_paths.append(full_path)
            # Check for retinopathy level in filename to infer label
            if '_eyepacs_0_' in filename or '_eyepacs_1_' in filename: # No Retinopathy (Nrdr) or mild (level 1)
                labels.append(0)
            elif any(f'_eyepacs_{i}_' in filename for i in [2, 3, 4]): # Retinopathy (Rdr) levels 2, 3, 4
                labels.append(1)
            else:
                print(f"Warning: Could not infer label for {filename}. Assigning default 0.")
                labels.append(0) # Default to 0 for unknown
else:
    print(f"Error: Test merge directory not found at {test_merge_path}")

# Convert lists to TensorFlow tensors
image_paths_tf = tf.constant(image_paths)
labels_tf = tf.constant(labels, dtype=tf.float32) # `label_mode="binary"` expects float32

# Function to load and preprocess a single image
def load_and_preprocess_image(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3) # Assuming JPEG, adjust if necessary
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE)) # Resize to model's expected input size
    # No normalization here, as `prep_eval` will handle `preprocess_input` later
    return img, label

# Create the test dataset
test_ds_raw = tf.data.Dataset.from_tensor_slices((image_paths_tf, labels_tf))
test_ds_raw = test_ds_raw.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds_raw = test_ds_raw.batch(BATCH_SIZE)
test_ds_raw = test_ds_raw.cache().prefetch(tf.data.AUTOTUNE) # Add cache and prefetch for performance

class_names = ['Nrdr', 'Rdr'] # Explicitly define since not inferred from directory structure
print("Class mapping:", class_names)

Found 4200 files belonging to 2 classes.
Found 1200 files belonging to 2 classes.
Class mapping: ['Nrdr', 'Rdr']


In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def prep_train(img, label):
    img = preprocess_input(img)
    return img, label

def prep_eval(img, label):
    img = preprocess_input(img)
    return img, label

print("Preparing training dataset...")
train_ds = train_ds_raw.map(prep_train, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
print("Training dataset prepared.")

print("Preparing validation dataset...")
val_ds   = val_ds_raw.map(prep_eval, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
print("Validation dataset prepared.")

print("Preparing test dataset...")
test_ds  = test_ds_raw.map(prep_eval, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
print("Test dataset prepared.")

all_labels = np.concatenate([y.numpy() for x, y in train_ds_raw], axis=0).flatten()
print("Class distribution:", np.bincount(all_labels.astype(int)))

Preparing training dataset...
Training dataset prepared.
Preparing validation dataset...
Validation dataset prepared.
Preparing test dataset...
Test dataset prepared.
Class distribution: [2100 2100]


In [ ]:
# def focal_loss(gamma=2.0, alpha=0.25):
#     def loss_fn(y_true, y_pred):
#         y_true = tf.cast(y_true, tf.float32)
#         epsilon = tf.keras.backend.epsilon()
#         y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

#         pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
#         alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

#         fl = -alpha_t * tf.pow(1 - pt, gamma) * tf.math.log(pt)
#         return tf.reduce_mean(fl)
#     return loss_fn

In [ ]:
def build_model():
    base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs)
    return model, base

model, base_model = build_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,586,177 (9.87 MB)

 Trainable params: 328,193 (1.25 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

phase 1 retraining

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks_p1 = [
    EarlyStopping(monitor="val_auc", patience=3, restore_best_weights=True, mode="max"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ModelCheckpoint(
        "/content/drive/MyDrive/EyePACS/best_model_p1.keras",
        monitor="val_auc", save_best_only=True, mode="max"
    )
]

history_p1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P1,
    callbacks=callbacks_p1
)

Epoch 1/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 60s 369ms/step - accuracy: 0.7157 - auc: 0.7767 - loss: 0.5756 - precision: 0.7272 - recall: 0.6905 - val_accuracy: 0.7750 - val_auc: 0.8482 - val_loss: 0.4833 - val_precision: 0.8235 - val_recall: 0.7000 - learning_rate: 0.0010
Epoch 2/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 64s 288ms/step - accuracy: 0.7595 - auc: 0.8315 - loss: 0.5042 - precision: 0.7815 - recall: 0.7205 - val_accuracy: 0.7700 - val_auc: 0.8538 - val_loss: 0.4910 - val_precision: 0.7396 - val_recall: 0.8333 - learning_rate: 0.0010
Epoch 3/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 36s 269ms/step - accuracy: 0.7662 - auc: 0.8427 - loss: 0.4858 - precision: 0.7838 - recall: 0.7352 - val_accuracy: 0.7692 - val_auc: 0.8542 - val_loss: 0.4806 - val_precision: 0.8519 - val_recall: 0.6517 - learning_rate: 0.0010
Epoch 4/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 38s 288ms/step - accuracy: 0.7807 - auc: 0.8561 - loss: 0.4671 - precision: 0.8006 - recall: 0.7476 - val_accuracy: 0.7800 - val_auc: 0.8596 - val_loss: 

phase 2 fine tuning

In [ ]:
import os
import tensorflow as tf

# Load the best model from Phase 1 training
print("Loading best model from Phase 1 ('best_model_p1.keras')...")
loaded_model_p1 = tf.keras.models.load_model(
    '/content/drive/MyDrive/EyePACS/best_model_p1.keras',
    custom_objects={'BinaryCrossentropy': tf.keras.losses.BinaryCrossentropy}
)

# Re-assign model to the loaded one
model = loaded_model_p1

# We also need to get the base_model part from the loaded model
# Assuming base_model is the first layer of the actual functional model after InputLayer
# Find the layer that matches the MobileNetV2 architecture
for layer in model.layers:
    if isinstance(layer, tf.keras.Model) and layer.name == 'mobilenetv2':
        base_model = layer
        break

print("Model loaded for Phase 2 fine-tuning.")

Loading best model from Phase 1 ('best_model_p1.keras')...
Model loaded for Phase 2 fine-tuning.


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks_p2 = [
    EarlyStopping(monitor="val_auc", patience=5, restore_best_weights=True, mode="max"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ModelCheckpoint(
        "/content/drive/MyDrive/EyePACS/best_model_final.keras",
        monitor="val_auc", save_best_only=True, mode="max"
    )
]

history_p2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P2,
    callbacks=callbacks_p2
)

Epoch 1/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 62s 386ms/step - accuracy: 0.8126 - auc: 0.8934 - loss: 0.4029 - precision: 0.8435 - recall: 0.7676 - val_accuracy: 0.7900 - val_auc: 0.8684 - val_loss: 0.4466 - val_precision: 0.8164 - val_recall: 0.7483 - learning_rate: 1.0000e-05
Epoch 2/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 41s 307ms/step - accuracy: 0.8121 - auc: 0.8969 - loss: 0.3986 - precision: 0.8441 - recall: 0.7657 - val_accuracy: 0.7908 - val_auc: 0.8692 - val_loss: 0.4458 - val_precision: 0.8133 - val_recall: 0.7550 - learning_rate: 1.0000e-05
Epoch 3/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 38s 292ms/step - accuracy: 0.8200 - auc: 0.9023 - loss: 0.3876 - precision: 0.8500 - recall: 0.7771 - val_accuracy: 0.7925 - val_auc: 0.8698 - val_loss: 0.4453 - val_precision: 0.8151 - val_recall: 0.7567 - learning_rate: 1.0000e-05
Epoch 4/15
132/132 ━━━━━━━━━━━━━━━━━━━━ 40s 300ms/step - accuracy: 0.8217 - auc: 0.9026 - loss: 0.3879 - precision: 0.8509 - recall: 0.7800 - val_accuracy: 0.7900 - val_auc: 0.8701 

test set evauation


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ── Best model load karo (Drive se) ────────────────────────────────────
best_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/EyePACS/best_model_final.keras"
)
print("Best model loaded!")

# ── TTA ────────────────────────────────────────────────────────────────
def tta_predict(model, img_batch):
    preds = [model.predict(img_batch, verbose=0)]
    preds.append(model.predict(tf.image.flip_left_right(img_batch), verbose=0))
    preds.append(model.predict(tf.image.flip_up_down(img_batch), verbose=0))
    preds.append(model.predict(tf.image.flip_left_right(tf.image.flip_up_down(img_batch)), verbose=0))
    return np.mean(preds, axis=0)

y_true, y_pred_prob = [], []
for imgs, labels in test_ds:
    preds = tta_predict(best_model, imgs).flatten()
    y_pred_prob.extend(preds)
    y_true.extend(labels.numpy().flatten())

y_true = np.array(y_true)
y_pred_prob = np.array(y_pred_prob)

# ── Threshold 0.25 ─────────────────────────────────────────────────────
THRESHOLD = 0.30
y_pred = (y_pred_prob >= THRESHOLD).astype(int)

# ── Metrics table ──────────────────────────────────────────────────────
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
metrics_df = pd.DataFrame(report_dict).transpose().round(4)
print(f"\nResults at threshold = {THRESHOLD}")
print(metrics_df)
print("\nOverall AUC:", round(roc_auc_score(y_true, y_pred_prob), 4))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm,
    index=[f"Actual_{c}" for c in class_names],
    columns=[f"Pred_{c}" for c in class_names])
print("\nConfusion Matrix:\n", cm_df)

Best model loaded!

Results at threshold = 0.3
              precision  recall  f1-score   support
Nrdr             0.8286  0.6767    0.7450  300.0000
Rdr              0.7268  0.8600    0.7878  300.0000
accuracy         0.7683  0.7683    0.7683    0.7683
macro avg        0.7777  0.7683    0.7664  600.0000
weighted avg     0.7777  0.7683    0.7664  600.0000

Overall AUC: 0.8698

Confusion Matrix:
              Pred_Nrdr  Pred_Rdr
Actual_Nrdr        203        97
Actual_Rdr          42       258


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

print(f"{'Threshold':<12}{'Recall(DR)':<13}{'Precision(DR)':<16}{'F1(DR)':<10}{'Accuracy'}")
print("-" * 55)

for t in np.arange(0.20, 0.55, 0.05):
    y_pred_t = (y_pred_prob >= t).astype(int)
    rec  = recall_score(y_true, y_pred_t, pos_label=1)
    prec = precision_score(y_true, y_pred_t, pos_label=1)
    f1   = f1_score(y_true, y_pred_t, pos_label=1)
    acc  = accuracy_score(y_true, y_pred_t)
    print(f"{t:<12.2f}{rec:<13.4f}{prec:<16.4f}{f1:<10.4f}{acc:.4f}")

Threshold   Recall(DR)   Precision(DR)   F1(DR)    Accuracy
-------------------------------------------------------
0.20        0.8633       0.6816          0.7618    0.7300
0.25        0.8267       0.7188          0.7690    0.7517
0.30        0.7900       0.7670          0.7783    0.7750
0.35        0.7600       0.7862          0.7729    0.7767
0.40        0.7200       0.8372          0.7742    0.7900
0.45        0.6967       0.8531          0.7670    0.7883
0.50        0.6700       0.8777          0.7599    0.7883
